In [3]:
import sys
sys.path.insert(0, "..")          # lets the notebook import from src/

import pandas as pd
from src.preprocess import (get_train_test_split, build_preprocessing_pipeline,
                            clean_and_engineer, NUMERIC_FEATURES, CATEGORICAL_FEATURES)

In [4]:
X_train, X_test, y_train, y_test = get_train_test_split("../data/telco_churn.csv")

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Churn rate train: %.2f%%" % (y_train.mean() * 100))
print("Churn rate test:  %.2f%%" % (y_test.mean() * 100))

Train: (5634, 20)  Test: (1409, 20)
Churn rate train: 26.54%
Churn rate test:  26.54%


In [5]:
cleaned = clean_and_engineer(X_train)
cleaned[["tenure", "TotalCharges", "NumAddOns", "AvgMonthlySpend"]].describe().round(2)

print(cleaned["MultipleLines"].unique())
print(cleaned["TechSupport"].unique())      # should be only ['No', 'Yes']
print(cleaned["TotalCharges"].isnull().sum())   # should be 0

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
0


In [6]:
tmp = cleaned.copy()
tmp["ChurnFlag"] = y_train.values
print(tmp.groupby("NumAddOns")["ChurnFlag"].mean().mul(100).round(1))

NumAddOns
0    20.9
1    45.8
2    36.2
3    27.8
4    23.0
5    12.5
6     4.7
Name: ChurnFlag, dtype: float64


In [7]:
prep = build_preprocessing_pipeline()
X_train_t = prep.fit_transform(X_train)   # fit + transform on train
X_test_t = prep.transform(X_test)         # transform only on test

print("Train matrix:", X_train_t.shape)
print("Test matrix: ", X_test_t.shape)

feature_names = prep.named_steps["encode_scale"].get_feature_names_out()
print(len(feature_names), "features")
print(list(feature_names)[:15])

Train matrix: (5634, 28)
Test matrix:  (1409, 28)
28 features
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'num__NumAddOns', 'num__AvgMonthlySpend', 'cat__gender_Male', 'cat__SeniorCitizen_Yes', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_Yes']


In [8]:
import numpy as np

assert not np.isnan(X_train_t).any() and not np.isnan(X_test_t).any()
assert X_train_t.shape[1] == X_test_t.shape[1]
assert "Churn" not in X_train.columns          # target not in features
assert "customerID" not in feature_names       # ID not used as a feature

# Scaler was fitted on train only, so train means are ~0 and test means are not exactly 0
num_idx = [i for i, n in enumerate(feature_names) if n.startswith("num__")]
print("Train numeric means:", X_train_t[:, num_idx].mean(axis=0).round(3))
print("Test numeric means: ", X_test_t[:, num_idx].mean(axis=0).round(3))
print("All checks passed")

Train numeric means: [-0. -0.  0.  0.  0.]
Test numeric means:  [-0.023 -0.028 -0.043 -0.054 -0.035]
All checks passed


In [9]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"Negative: {neg}, Positive: {pos}, scale_pos_weight: {scale_pos_weight:.2f}")

Negative: 4139, Positive: 1495, scale_pos_weight: 2.77
